# 🧹 MAINTENANCE (DESTRUKTIF) — Strip `3DCNN/dataset` dari SELURUH history git

**Jalankan HANYA di Colab** (butuh disk ~30GB + langsung ke github.com). **JANGAN** di container lokal.

## ⚠️ PERINGATAN
Notebook ini **menulis ulang seluruh history** repo untuk membuang `3DCNN/dataset` (reclaim ~12.8 GB),
lalu **force-push semua branch & tag**. Akibat:
- **Semua commit SHA berubah.**
- **Pull Request lama jadi invalid** (base/head SHA hilang).
- **Semua clone lama harus di-clone ulang** (history lama tak kompatibel).

## Prasyarat (WAJIB sebelum lanjut)
1. Dataset versi terbaru **sudah aman** sebagai aset GitHub Release (tag `data-v8`) —
   notebook ini memverifikasinya (sel 2). Jangan hapus history sebelum data aman di luar git.
2. Tidak ada PR penting yang sedang berjalan (atau Anda siap me-rebase/buka ulang).
3. `GITHUB_TOKEN` (PAT scope `repo`) tersimpan di Colab `userdata`.

Alur: precondition → backup (bundle → Release) → `git filter-repo` (mirror) → force-push → verifikasi.



## 0. Konfigurasi


In [ ]:
from google.colab import userdata
from pathlib import Path
import subprocess, sys, os, time

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
REPO_SLUG    = 'RZulfikri/Thesis'
REPO_URL     = f'https://{GITHUB_TOKEN}@github.com/{REPO_SLUG}.git'
DATA_TAG     = 'data-v8'                 # Release berisi dataset (precondition)
# Buang hanya FILE DATA di bawah 3DCNN/dataset (ply/npy/json) dari SELURUH history,
# tapi PERTAHANKAN 3DCNN/dataset/README.md (penjelasan data-via-Release). Lebih bedah
# daripada menghapus seluruh folder.
STRIP_GLOBS  = ['3DCNN/dataset/**/*.ply',
                '3DCNN/dataset/**/*.npy',
                '3DCNN/dataset/**/*.json']
BACKUP_TAG   = 'backup-prestrip-v8'      # Release tujuan upload bundle backup
WORK         = Path('/content/strip_work')
MIRROR       = WORK / 'thesis_mirror.git'

# GERBANG KEAMANAN: ubah ke True HANYA bila Anda paham konsekuensinya.
CONFIRM_DESTRUCTIVE = False

WORK.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, '/content')  # release_assets diunduh ke /content di sel 1
print('Config OK. CONFIRM_DESTRUCTIVE =', CONFIRM_DESTRUCTIVE)


## 1. Ambil helper `release_assets.py` (dari repo, tanpa clone penuh)


In [ ]:
# Unduh hanya 1 file helper via raw API (hindari clone 12.8GB).
import urllib.request, json, base64
_u = f'https://api.github.com/repos/{REPO_SLUG}/contents/3DRegistration/release_assets.py?ref=colab'
_r = urllib.request.Request(_u, headers={'Authorization': f'Bearer {GITHUB_TOKEN}',
                                         'Accept': 'application/vnd.github+json',
                                         'User-Agent': 'strip-maint'})
_d = json.loads(urllib.request.urlopen(_r).read())
open('/content/release_assets.py','wb').write(base64.b64decode(_d['content']))
import importlib, release_assets as ra; importlib.reload(ra)
print('release_assets.py siap.')



## 2. PRECONDITION — Pastikan Release `data-v8` lengkap & konsisten
Verifikasi release ada, punya MANIFEST, dan **semua part** (nama+ukuran) cocok dengan manifest.
(Tidak mengunduh penuh; cukup metadata aset.) **Gagal di sini = STOP, jangan rewrite.**


In [ ]:
rel = ra.get_release(REPO_SLUG, DATA_TAG, GITHUB_TOKEN)
assert rel, f'Release {DATA_TAG} TIDAK ADA — buat & upload dataset dulu (pack_dataset_release + upload).'
assets = {a['name']: a for a in rel.get('assets', [])}
man_name = next((n for n in assets if n.startswith('MANIFEST_')), None)
assert man_name, f'MANIFEST tidak ada di release {DATA_TAG}.'

# unduh manifest kecil saja
ra.download_asset(REPO_SLUG, DATA_TAG, man_name, WORK / man_name, GITHUB_TOKEN)
manifest = json.loads((WORK / man_name).read_text())
print('manifest:', man_name, '| file_count =', manifest['file_count'], '| parts =', len(manifest['parts']))

missing, mismatch = [], []
for pm in manifest['parts']:
    a = assets.get(pm['name'])
    if not a: missing.append(pm['name'])
    elif a['size'] != pm['size']: mismatch.append((pm['name'], a['size'], pm['size']))
assert not missing,  f'Part HILANG di release: {missing}'
assert not mismatch, f'Ukuran part tak cocok manifest: {mismatch}'
print(f'PRECONDITION OK — {len(manifest["parts"])} part cocok manifest. Dataset aman di Release.')



## 3. Mirror-clone repo (state PRA-rewrite) + BACKUP bundle → Release
Mirror clone = salinan semua ref (branch+tag). Lalu buat `git bundle --all` dan upload sbg aset
Release `backup-prestrip-v8` (pemulihan total bila perlu).


In [ ]:
import shutil
if MIRROR.exists(): shutil.rmtree(MIRROR)
print('Mirror-clone (≈12.8GB, beberapa menit) ...')
t0=time.time()
subprocess.run(['git','clone','--mirror',REPO_URL,str(MIRROR)], check=True)
print(f'  selesai {time.time()-t0:.0f}s')

bundle = WORK / 'backup_prestrip_v8.bundle'
print('Buat bundle --all ...')
subprocess.run(['git','-C',str(MIRROR),'bundle','create',str(bundle),'--all'], check=True)
print(f'  bundle: {bundle.stat().st_size/1e9:.2f} GB')

# bundle bisa >2GB → pakai pack_dataset split? Tidak; bundle satu file. Bila >1.9GB, split manual.
import math
LIMIT=1900*1024*1024
to_upload=[]
if bundle.stat().st_size > LIMIT:
    print('  bundle > 1.9GB → split ...')
    subprocess.run(['split','-b',str(LIMIT),'-d','-a','2',str(bundle),str(bundle)+'.part'], check=True)
    to_upload=sorted(str(p) for p in WORK.glob(bundle.name+'.part*'))
else:
    to_upload=[str(bundle)]

ra.create_or_get_release(REPO_SLUG, BACKUP_TAG, GITHUB_TOKEN,
                         name='Backup pra-strip (full history bundle)',
                         body='git bundle --all sebelum filter-repo strip 3DCNN/dataset. '
                              'Restore: cat part* > b.bundle && git clone b.bundle restore', prerelease=True)
for f in to_upload:
    ra.upload_asset(REPO_SLUG, BACKUP_TAG, f, GITHUB_TOKEN)
print('BACKUP terupload ke Release', BACKUP_TAG)



## 4. `git filter-repo` — buang `3DCNN/dataset` dari SEMUA commit
Butuh `CONFIRM_DESTRUCTIVE = True`. Belum push apa pun di sini (hanya menulis ulang mirror lokal).


In [ ]:
assert CONFIRM_DESTRUCTIVE, 'Set CONFIRM_DESTRUCTIVE=True di sel 0 untuk melanjutkan (DESTRUKTIF).'
subprocess.run([sys.executable,'-m','pip','install','-q','git-filter-repo'], check=True)

print('Ukuran mirror SEBELUM:')
subprocess.run(['git','-C',str(MIRROR),'count-objects','-vH'])

print('\nfilter-repo (buang file data', STRIP_GLOBS, 'dari semua history) ...')
t0=time.time()
# --invert-paths: hapus union semua --path-glob (ply/npy/json) → README.md tetap.
# --force: filter-repo menolak repo non-fresh; mirror OK.
cmd=['git','-C',str(MIRROR),'filter-repo','--invert-paths','--force']
for g in STRIP_GLOBS:
    cmd += ['--path-glob', g]
r=subprocess.run(cmd)
assert r.returncode==0, 'filter-repo gagal'
print(f'  selesai {time.time()-t0:.0f}s')

# CATATAN: filter-repo SUDAH otomatis reflog-expire + repack + prune di akhir → ukuran
# sudah turun. Cukup gc RINGAN (prune sisa). JANGAN pakai '--aggressive': pada repo besar
# bisa 10-30 mnt & DIAM TOTAL (tampak seperti hang) tanpa manfaat berarti.
print('gc ringan (prune sisa) ...')
subprocess.run(['git','-C',str(MIRROR),'gc','--prune=now'])
print('\nUkuran mirror SESUDAH:')
subprocess.run(['git','-C',str(MIRROR),'count-objects','-vH'])

# sanity: tak ada lagi blob data (ply/npy) yg tersisa di history
chk=subprocess.run(['git','-C',str(MIRROR),'rev-list','--all','--objects'],
                   capture_output=True, text=True)
left=[l for l in chk.stdout.splitlines()
      if l.endswith('.ply') or l.endswith('.npy')]
left=[l for l in left if '3DCNN/dataset/' in l]
print('blob data dataset tersisa:', len(left), '(harus 0)')
assert not left, f'Masih ada blob data: contoh {left[:3]} — STOP, jangan push.'
# README harus masih ada
chk2=subprocess.run(['git','-C',str(MIRROR),'log','--all','--oneline','--','3DCNN/dataset/README.md'],
                    capture_output=True, text=True)
print('README.md masih di history:', bool(chk2.stdout.strip()))


## 5. FORCE-PUSH semua branch & tag (titik tak-bisa-kembali)
`--all` + `--tags` (BUKAN `--mirror`, agar tag Release `data-v8`/`backup-prestrip-v8` yg dibuat
setelah mirror clone TIDAK terhapus). Butuh `CONFIRM_DESTRUCTIVE=True`.


In [ ]:
assert CONFIRM_DESTRUCTIVE, 'Set CONFIRM_DESTRUCTIVE=True dulu.'
# filter-repo menghapus remote 'origin' → tambah lagi
subprocess.run(['git','-C',str(MIRROR),'remote','remove','origin'], capture_output=True)
subprocess.run(['git','-C',str(MIRROR),'remote','add','origin',REPO_URL], check=True)

print('force-push --all ...')
r1=subprocess.run(['git','-C',str(MIRROR),'push','--force','--all','origin'])
print('force-push --tags ...')
r2=subprocess.run(['git','-C',str(MIRROR),'push','--force','--tags','origin'])
assert r1.returncode==0 and r2.returncode==0, 'push gagal — repo remote belum berubah; cek error.'
print('\n✅ HISTORY DITULIS ULANG & TER-PUSH. Semua SHA berubah; re-clone diperlukan.')



## 6. Verifikasi pasca-rewrite (clone segar single-branch)
Konfirmasi repo kini ramping & report/eval masih ada.


In [ ]:
import shutil
V = WORK / 'verify_clone'
if V.exists(): shutil.rmtree(V)
t0=time.time()
subprocess.run(['git','clone','--single-branch','--branch','colab','--depth','1',REPO_URL,str(V)], check=True)
print(f'clone colab (shallow) {time.time()-t0:.0f}s')
subprocess.run(['git','-C',str(V),'count-objects','-vH'])
import os
print('\nreport ada :', (V/'3DCNN'/'result_docs').exists())
print('analysis ada:', (V/'3DCNN'/'analysis').exists())
print('dataset kosong (selain README):',
      sorted(p.name for p in (V/'3DCNN'/'dataset').glob('*')) if (V/'3DCNN'/'dataset').exists() else 'dir tak ada')
print('\nSelesai. Dataset mengalir dari Release', DATA_TAG, '(sel Data Bootstrap di notebook training).')

